# 01 数据集概览与 grouped nested-CV 验证

本模块描述二分类数据集、负类亚型、来源录音与检测分数，并验证 grouped nested-CV 的来源隔离。所有解释使用中文，图表标签使用英文。生成结果写入 `workspace/paper_analysis/01_dataset_overview/`。

In [1]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "project_config.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project repository.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_config import WORKSPACE_DIR

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE_DIR / "matplotlib_cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

def require_columns(frame, required, name):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

def save_figure(fig, stem):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.close(fig)

def write_latex(frame, name, **kwargs):
    (OUTPUT_DIR / name).write_text(frame.to_latex(index=False, **kwargs), encoding="utf-8")

from project_config import BINARY_LABELS_PATH, FOLDS_PATH, FOLD_BALANCE_REPORT_PATH

RANDOM_SEED = 2021
BOOTSTRAP_ITERATIONS = 5000  # 本模块不执行 bootstrap；保留统一论文配置。
OUTPUT_DIR = WORKSPACE_DIR / "paper_analysis" / "01_dataset_overview"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUTS = [BINARY_LABELS_PATH, FOLDS_PATH, FOLD_BALANCE_REPORT_PATH]
missing_inputs = [str(path) for path in INPUTS if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(f"Missing required inputs: {missing_inputs}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Project root: /path/to/strom_petrels
Output directory: /path/to/strom_petrels/workspace/paper_analysis/01_dataset_overview


## 数据集构成

数据层级为三秒片段；`source_file` 是用于分组拆分和后续聚类推断的独立录音单位。检测分数仅描述样本选择，不等同于模型的生物学概率。

In [2]:
labels = pd.read_csv(BINARY_LABELS_PATH)
require_columns(labels, {
    "clip_id", "label", "target_label", "negative_type", "species",
    "source_file", "score"
}, "binary labels")

if labels["clip_id"].duplicated().any():
    raise AssertionError("clip_id must be unique in binary_labels.csv")
if set(labels["target_label"].dropna().astype(int).unique()) != {0, 1}:
    raise AssertionError("target_label must contain exactly 0 and 1")

class_summary = (
    labels.groupby(["label", "target_label"], dropna=False)
    .agg(clips=("clip_id", "size"), source_recordings=("source_file", "nunique"),
         median_detection_score=("score", "median"))
    .reset_index()
)
negative_summary = (
    labels.loc[labels["target_label"] == 0]
    .groupby("negative_type", dropna=False)
    .agg(clips=("clip_id", "size"), source_recordings=("source_file", "nunique"))
    .reset_index()
)
dataset_summary = pd.DataFrame({
    "statistic": ["clips", "positive_clips", "negative_clips", "source_recordings", "negative_subtypes"],
    "value": [len(labels), int((labels.target_label == 1).sum()), int((labels.target_label == 0).sum()),
              labels.source_file.nunique(), labels.negative_type.dropna().nunique()],
})

class_summary.to_csv(OUTPUT_DIR / "class_summary.csv", index=False)
negative_summary.to_csv(OUTPUT_DIR / "negative_subtype_summary.csv", index=False)
dataset_summary.to_csv(OUTPUT_DIR / "dataset_summary.csv", index=False)
write_latex(class_summary, "dataset_summary.tex", float_format="%.3f")
display(dataset_summary)
display(class_summary)
display(negative_summary)

,statistic,value
0,clips,2000
1,positive_clips,1000
2,negative_clips,1000
3,source_recordings,26
4,negative_subtypes,2


,label,target_label,clips,source_recordings,median_detection_score
0,not_storm_petrel,0,1000,26,0.602522
1,storm_petrel,1,1000,26,0.999958


,negative_type,clips,source_recordings
0,environment_noise,500,23
1,other_bird,500,20


In [3]:
source_counts = labels.groupby(["source_file", "label"]).size().unstack(fill_value=0)
source_counts = source_counts.sort_values(source_counts.columns.tolist(), ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
class_summary.set_index("label")["clips"].plot.bar(ax=axes[0, 0], color=["#6c757d", "#d95f02"])
axes[0, 0].set(title="Class distribution", xlabel="Class", ylabel="Three-second clips")
axes[0, 0].tick_params(axis="x", rotation=0)

negative_summary.set_index("negative_type")["clips"].plot.bar(ax=axes[0, 1], color="#4c78a8")
axes[0, 1].set(title="Negative subtype distribution", xlabel="Negative subtype", ylabel="Clips")
axes[0, 1].tick_params(axis="x", rotation=20)

source_counts.plot.bar(stacked=True, ax=axes[1, 0], color=["#6c757d", "#d95f02"], width=0.85)
axes[1, 0].set(title="Class composition by source recording", xlabel="Source recording", ylabel="Clips")
axes[1, 0].tick_params(axis="x", labelsize=6, rotation=90)

for target, group in labels.groupby("target_label"):
    axes[1, 1].hist(group["score"].dropna(), bins=30, alpha=0.55,
                    label="Positive" if target == 1 else "Negative")
axes[1, 1].set(title="Detection-score distribution", xlabel="Input detection score", ylabel="Clips")
axes[1, 1].legend()
save_figure(fig, "dataset_distributions")

## Outer-fold 来源隔离与覆盖

每个 outer fold 内训练来源和测试来源必须完全不相交；五个 outer test 合并后，每个片段必须恰好出现一次。这些是数据拆分的结构不变量，而不是模型性能结论。

In [4]:
folds = pd.read_csv(FOLDS_PATH, dtype={"outer_fold": str, "inner_fold": str})
require_columns(folds, {"clip_id", "source_file", "target_label", "negative_type", "split", "outer_fold"}, "nested-CV folds")
folds["outer_fold"] = folds["outer_fold"].str.zfill(2)

validation_rows = []
for outer_fold, group in folds.groupby("outer_fold", sort=True):
    train = group[group["split"] == "outer_train"]
    test = group[group["split"] == "outer_test"]
    overlap = sorted(set(train.source_file) & set(test.source_file))
    inner_overlap_count = 0
    inner_folds = sorted(train["inner_fold"].dropna().unique())
    for inner_fold in inner_folds:
        inner_validation_sources = set(train.loc[train.inner_fold == inner_fold, "source_file"])
        inner_training_sources = set(train.loc[train.inner_fold != inner_fold, "source_file"])
        inner_overlap_count += len(inner_validation_sources & inner_training_sources)
    validation_rows.append({
        "outer_fold": outer_fold,
        "train_clips": len(train), "test_clips": len(test),
        "train_sources": train.source_file.nunique(), "test_sources": test.source_file.nunique(),
        "outer_overlapping_sources": len(overlap),
        "inner_folds_checked": len(inner_folds),
        "inner_overlapping_sources": inner_overlap_count,
    })
    if overlap:
        raise AssertionError(f"Source leakage in outer fold {outer_fold}: {overlap}")
    if inner_overlap_count:
        raise AssertionError(f"Source leakage in inner folds nested under outer fold {outer_fold}")

outer_test = folds[folds["split"] == "outer_test"].copy()
assignment_counts = outer_test.groupby("clip_id").size()
if len(assignment_counts) != len(labels) or not assignment_counts.eq(1).all():
    raise AssertionError("Every clip must occur exactly once across outer test folds")
if set(assignment_counts.index) != set(labels.clip_id):
    raise AssertionError("Outer-test clip IDs do not match binary_labels.csv")

fold_validation = pd.DataFrame(validation_rows)
fold_validation["all_checks_passed"] = (
    fold_validation["outer_overlapping_sources"].eq(0)
    & fold_validation["inner_overlapping_sources"].eq(0)
)
fold_validation.to_csv(OUTPUT_DIR / "fold_validation.csv", index=False)
write_latex(fold_validation, "fold_validation.tex")
display(fold_validation)
print(f"Validated {len(assignment_counts):,} unique outer-test clips; every clip appears once.")

,outer_fold,train_clips,test_clips,train_sources,test_sources,outer_overlapping_sources,inner_folds_checked,inner_overlapping_sources,all_checks_passed
0,01,1562,438,23,3,0,3,0,True
1,02,1640,360,19,7,0,3,0,True
2,03,1628,372,22,4,0,3,0,True
3,04,1562,438,23,3,0,3,0,True
4,05,1608,392,17,9,0,3,0,True


Validated 2,000 unique outer-test clips; every clip appears once.


In [5]:
fold_report = pd.read_csv(FOLD_BALANCE_REPORT_PATH, dtype={"outer_fold": str, "inner_fold": str})
require_columns(fold_report, {"outer_fold", "inner_fold", "split", "positive", "negative", "source_files"}, "fold balance report")
fold_report["outer_fold"] = fold_report["outer_fold"].str.zfill(2)
outer_balance = fold_report[(fold_report["split"] == "test") & fold_report["inner_fold"].isna()].copy()
outer_balance.to_csv(OUTPUT_DIR / "outer_fold_balance.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
outer_balance.set_index("outer_fold")[["positive", "negative"]].plot.bar(
    ax=axes[0], color=["#d95f02", "#6c757d"]
)
axes[0].set(title="Outer-test class balance", xlabel="Outer fold", ylabel="Clips")
axes[0].tick_params(axis="x", rotation=0)

outer_balance.set_index("outer_fold")["source_files"].plot.bar(ax=axes[1], color="#4c78a8")
axes[1].set(title="Outer-test source recordings", xlabel="Outer fold", ylabel="Source recordings")
axes[1].tick_params(axis="x", rotation=0)
save_figure(fig, "outer_fold_balance")

print("Structural validation complete: no outer-fold source leakage and one outer-test assignment per clip.")

Structural validation complete: no outer-fold source leakage and one outer-test assignment per clip.
